# Two-stage comparison across time-frequency decompositions

Compares two-stage (N-vs-tremor → PD-vs-ET) classifiers built on different
**time-frequency decompositions**, chosen by the model-free separability study:
STFT-256 (best 3-class), HHT-8IMF (best PD-vs-ET), CWT, and a **hybrid**
(STFT-256 for stage 1, HHT-8IMF for stage 2). Honest leave-one-patient-out with
subject bootstrap CIs. CPU-only.

Note: HHT feature extraction takes ~1 min (transform is slow, but computed
once).

## 1. Setup

In [ ]:
import sys, os
if os.path.basename(os.getcwd())=="pdetn": os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, matplotlib.pyplot as plt
from tremor.data import CLASS_NAMES
from tremor.quaternion_data import load_quaternion_recordings
from pdetn.separability import patient_decomp_features
from pdetn.model import TwoStageClassifier, FlatClassifier, HybridTwoStage
from pdetn.evaluate import evaluate, evaluate_hybrid, print_result
DATA_ROOT="Data"; ACTION="OUT"
recs = load_quaternion_recordings(DATA_ROOT, action=ACTION, mode="angular_velocity")
print(len(recs), "recordings,", len(set(r.subject for r in recs)), "patients")

## 2. Per-patient decomposition features\nEach method reduced to mean+std over time, aggregated per patient. HHT is the slow one (~1 min).

In [ ]:
feat = {
  "stft256": patient_decomp_features(recs,"stft",nperseg=256,nfft=256,noverlap=192),
  "cwt":     patient_decomp_features(recs,"cwt",cwt_w0=6.0,cwt_freq_step=0.5),
  "hht8":    patient_decomp_features(recs,"hht",hht_max_imfs=8),
}
pats = feat["stft256"][2]; y = feat["stft256"][1]; subj = pats
assert all((feat[k][2]==pats).all() for k in feat)
print("features:", {k:v[0].shape for k,v in feat.items()})

## 3. Two-stage per single decomposition (tuned ET threshold)

In [ ]:
results={}
for name,(X,_,_) in feat.items():
    r=evaluate(lambda: TwoStageClassifier("logreg","logreg",tune_et_threshold=True), X,y,subj,n_boot=1000)
    results[f"2stage_{name}"]=r; print_result(f"2stage {name}", r)

## 4. Hybrid two-stage: STFT-256 (N-vs-tremor) + HHT-8IMF (PD-vs-ET)\nUse each representation where it separates best.

In [ ]:
X1=feat["stft256"][0]; X2=feat["hht8"][0]
rh=evaluate_hybrid(lambda: HybridTwoStage("logreg","logreg",tune_et_threshold=True), X1,X2,y,subj,n_boot=1000)
results["HYBRID_stft256+hht8"]=rh; print_result("HYBRID stft256->hht8", rh)

## 5. Comparison table + chart

In [ ]:
ref = {"deep_STFT(ref)":{"macro_f1":0.63,"per_class_f1":{"ET":0.47}},
       "biomarker_2stage(ref)":{"macro_f1":0.582,"per_class_f1":{"ET":0.324}}}
print(f"{'config':>24}{'macroF1':>9}{'ET_F1':>8}{'PDvsET':>8}{'Nvstre':>8}")
for k,r in results.items():
    print(f"{k:>24}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}{r.get('pd_vs_et_acc',float('nan')):>8.2f}{r.get('n_vs_tremor_acc',float('nan')):>8.2f}")
for k,r in ref.items(): print(f"{k:>24}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}")
labels=list(results)+list(ref); macro=[results.get(k,ref.get(k))['macro_f1'] for k in labels]
etf1=[(results.get(k,ref.get(k)))['per_class_f1']['ET'] for k in labels]
xi=np.arange(len(labels)); w=0.4
fig,ax=plt.subplots(figsize=(11,4)); ax.bar(xi-w/2,macro,w,label="macro-F1",color="#2c7fb8"); ax.bar(xi+w/2,etf1,w,label="ET-F1",color="#d95f02")
ax.set_xticks(xi); ax.set_xticklabels(labels,rotation=30,ha="right"); ax.axhline(1/3,ls="--",c="gray",lw=1,label="chance")
ax.set_ylabel("F1"); ax.set_title("Two-stage across TF decompositions (OUT)"); ax.legend(); plt.tight_layout(); plt.show()

## 6. Confusion matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
keys=list(results); fig,ax=plt.subplots(1,len(keys),figsize=(4*len(keys),3.6))
for a,k in zip(np.atleast_1d(ax),keys):
    ConfusionMatrixDisplay(np.array(results[k]['confusion_matrix']),display_labels=CLASS_NAMES).plot(ax=a,colorbar=False)
    a.set_title(f"{k}\nmF1 {results[k]['macro_f1']:.2f}",fontsize=9)
plt.tight_layout(); plt.show()

## 6.5 Temporal-spatial features (3-sensor arm geometry)

Add the spatial axis — hand/lower_arm/upper_arm tremor propagation (per-sensor
power, distal→proximal gradients, cross-sensor coherence & phase) — on top of
the time-frequency transform. Compare TF vs spatial vs **TF+spatial**, per
condition. Result: TF+spatial on OUT gives the best ET-F1 (0.42); WING gives the
best macro-F1 (0.67); spatial rescues the TF-poor REST condition.

In [ ]:
from collections import defaultdict
from tremor.quaternion_data import load_quaternion_recordings
from pdetn.separability import patient_decomp_features
from pdetn.spatial_features import spatial_features, SPATIAL_FEATURE_NAMES
def patient_spatial(recs):
    per=defaultdict(list); lab={}
    for r in recs:
        per[r.subject].append([spatial_features(r.x)[f] for f in SPATIAL_FEATURE_NAMES]); lab[r.subject]=r.y
    pats=sorted(per); return (np.array([np.mean(per[p],axis=0) for p in pats]),
                              np.array([lab[p] for p in pats]), np.array(pats))
print(f"{'cond':>5}{'featureset':>14}{'macroF1':>9}{'ET_F1':>8}{'PDvsET':>8}")
for cond in ["OUT","REST","WING"]:
    rc=load_quaternion_recordings(DATA_ROOT,action=cond,mode="angular_velocity")
    Xtf,y,subj=patient_decomp_features(rc,"stft",nperseg=256,nfft=256,noverlap=192)
    Xsp,_,_=patient_spatial(rc)
    for nm,X in [("TF",Xtf),("spatial",Xsp),("TF+spatial",np.concatenate([Xtf,Xsp],1))]:
        r=evaluate(lambda: TwoStageClassifier("logreg","logreg",tune_et_threshold=True), X,y,subj,n_boot=500)
        print(f"{cond:>5}{nm:>14}{r['macro_f1']:>9.3f}{r['per_class_f1']['ET']:>8.3f}{r['pd_vs_et_acc']:>8.2f}")

## 7. Findings (see reports/decomposition_study.md)

- **STFT-256 two-stage is best overall** (macro-F1 0.651, ET-F1 0.378) — tuning the STFT window (256 vs default 128) gave a real gain over the biomarker feature set (0.582/0.324).
- **HHT-8 / hybrid win PD-vs-ET *accuracy*** (0.78–0.79, matching the separability study) but lower ET-F1 — ET-F1 also depends on stage-1 routing and the minority threshold.
- Decomposition study (model-free): STFT-256 best 3-class, HHT-7/8 best PD-vs-ET, CWT all-rounder; SST and feature fusion do not beat them.
- Ceiling is still ~16 ET subjects — external data (PADS) remains the real lever.